# OCR Evaluation · Полный пайплайн

Объединённый ноутбук для Google Colab. Содержит все этапы:
1. **Setup & Dataset** — установка зависимостей, загрузка OmniDocBench, сохранение subset
2. **DeepSeek-OCR** — инференс модели `deepseek-ai/DeepSeek-OCR`
3. **mPLUG-DocOwl 2** — инференс модели `mPLUG/DocOwl2`
4. **olmOCR** — инференс модели `allenai/olmOCR-7B-0225-preview`
5. **MonkeyOCR** — инференс модели `echo840/MonkeyOCR`

> **Совет:** В Colab используйте GPU runtime (T4/L4/A100). Запускайте секции последовательно — каждая модель освобождает GPU перед загрузкой следующей.

---
# Часть 1 · Подготовка среды и загрузка OmniDocBench

Этот раздел готовит среду к экспериментам:
1. Устанавливает зависимости из `requirements.txt`.
2. Скачивает датасет **OmniDocBench v1.6** (`opendatalab/OmniDocBench` на HuggingFace).
3. Загружает разметку, фильтрует страницы типа `academic_literature` (научные статьи arXiv-стиля).
4. Показывает превью первой страницы и её ground-truth.
5. Сохраняет subset в `data/subset.json` — все модели используют один и тот же список страниц.

In [ ]:
# === Colab / Kaggle bootstrap =================================================
# В Colab клонируем репозиторий проекта (предполагается, что код выложен
# в GitHub) и переходим в его корень. Для локального запуска просто
# проверьте, что текущая рабочая директория — корень ocr_eval/.
import os, sys, pathlib

# Клонируем только если ещё нет (защита от повторного запуска)
if not pathlib.Path('ocr_eval').exists():
    !git clone https://github.com/AStrateg2509/ocr_eval.git

os.chdir('ocr_eval')
sys.path.insert(0, 'src')
print("CWD =", os.getcwd())

Cloning into 'ocr_eval'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 46 (delta 22), reused 42 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 30.31 KiB | 4.33 MiB/s, done.
Resolving deltas: 100% (22/22), done.
CWD = /content/ocr_eval


In [ ]:
# 1. Проверка системных требований
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"CUDNN: {torch.backends.cudnn.version()}")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA: 12.8
CUDNN: 91002


In [ ]:
# First, check your CUDA version
!nvcc --version

# Check GPU model
!nvidia-smi

# Установите PyTorch
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Mon Apr 27 11:29:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8       

In [ ]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 13.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Flash-attn для PyTorch
!pip install --no-deps --force-reinstall https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.1.post4/flash_attn-2.7.1.post4+cu12torch2.6cxx11abiFALSE-cp312-cp312-linux_x86_64.whl --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.4/187.4 MB 6.0 MB/s eta 0:00:00


In [ ]:
from src.dataset_loader import download_omnidocbench, load_omnidocbench, iter_pages

DATA_ROOT = 'data/OmniDocBench'
download_omnidocbench(DATA_ROOT, source='huggingface')
print('OK, датасет в', DATA_ROOT)

In [ ]:
# Берём 100 случайных страниц-научных публикаций для экспериментов
items = load_omnidocbench(
    root=DATA_ROOT,
    page_types=['academic_literature'],
    languages=['english'],
    subset_size=5,
    seed=42,
)
print(f'Отобрано страниц: {len(items)}')
items[0].to_dict() if items else 'empty'

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

first = items[0]
img = Image.open(Path(DATA_ROOT) / first.image_path).convert('RGB')
fig, ax = plt.subplots(figsize=(8, 11))
ax.imshow(img); ax.axis('off')
ax.set_title(f'{first.page_id}  ·  {first.page_type}')
plt.show()

In [ ]:
# Сохраняем список выбранных страниц.
# Все 4 блока инференса используют один и тот же subset — так результаты сравнимы.
import json, pathlib
subset_path = pathlib.Path('data/subset.json')
subset_path.parent.mkdir(parents=True, exist_ok=True)
subset_path.write_text(
    json.dumps([x.to_dict() for x in items], ensure_ascii=False),
    encoding='utf-8',
)
print('сохранено:', subset_path, subset_path.stat().st_size, 'байт')

---
# Часть 2 · DeepSeek-OCR

**DeepSeek-OCR** — открытая мульти-модальная модель (≈3B параметров) от DeepSeek-AI; поддерживает grounding-промпт и markdown-вывод. Чекпоинт: [`deepseek-ai/DeepSeek-OCR`](https://huggingface.co/deepseek-ai/DeepSeek-OCR).

In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord

cfg = load_config('configs/deepseek_ocr.yaml')
print(gpu_info())
cfg

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_REPO = cfg['model']['hf_repo']
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
    # attn_implementation="eager",  # <-- Ключевой параметр для исправления
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

In [ ]:
import os, time, traceback
from pathlib import Path
from PIL import Image

out_dir = Path(cfg['output']['results_dir'])
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'predictions.jsonl'
done = already_processed_ids(out_path)
print(f'уже обработано: {len(done)} / {len(subset)}')

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done:
            continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists():
            continue
        rec = PredictionRecord(page_id=gt.page_id, model='deepseek_ocr')
        try:
            with Timer('infer') as t:
                tmp_out = out_dir / f'{gt.page_id}'
                tmp_out.mkdir(parents=True, exist_ok=True)
                result_md = model.infer(
                    tokenizer,
                    prompt=cfg['inference']['prompt'],
                    image_file=str(img_path),
                    output_path=str(tmp_out),
                    base_size=cfg['inference']['base_size'],
                    image_size=cfg['inference']['image_size'],
                    crop_mode=cfg['inference']['crop_mode'],
                    save_results=False,
                    test_compress=cfg['inference']['test_compress'],
                )
            rec.full_text = result_md if isinstance(result_md, str) else str(result_md)
            rec.raw_output = rec.full_text
            rec.inference_time_s = t.elapsed
            w.write(rec.to_dict())      # ✅ пишем ТОЛЬКО при успехе
            done.add(gt.page_id)        # ✅ обновляем done в процессе
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
            # ❌ НЕ пишем в файл — страница останется необработанной
            #    и будет повторно обработана при следующем запуске

In [ ]:
# Превью одного результата
from src.utils import read_jsonl
preds = read_jsonl(out_path)
print(len(preds), 'записей')
print(preds[1]['full_text'][:1000])

In [ ]:
# Освобождаем GPU перед следующей моделью
del model; cuda_free(); print(gpu_info())

Tesla T4 | used 0.01 / 14.6 GiB


---
# Часть 3 · mPLUG-DocOwl 2

**mPLUG-DocOwl 2** — модель Alibaba для понимания документов с shape-adaptive cropping. Чекпоинт: [`mPLUG/DocOwl2`](https://huggingface.co/mPLUG/DocOwl2).

In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord

cfg = load_config('configs/mplug_docowl.yaml')
cfg

{'model': {'name': 'mplug_docowl',
  'hf_repo': 'mPLUG/DocOwl2',
  'trust_remote_code': True,
  'torch_dtype': 'bfloat16',
  'device_map': 'auto'},
 'inference': {'prompt': 'Parse this document. Output structure as markdown including tables in HTML.',
  'high_resolution': True,
  'resolution': 1024,
  'max_new_tokens': 4096,
  'do_sample': False,
  'num_beams': 1},
 'dataset': {'name': 'omnidocbench', 'subset_size': 100, 'split': 'test'},
 'output': {'results_dir': 'results/mplug_docowl', 'save_format': 'jsonl'}}

In [ ]:
!pip install icecream

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_REPO = cfg['model']['hf_repo']
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

visual_compressor.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/mPLUG/DocOwl2:
- visual_compressor.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


visual_encoder.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/mPLUG/DocOwl2:
- visual_encoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_llama2_mam.py: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/17.1G [00:00<?, ?B/s]

Some weights of MPLUGDocOwl2 were not initialized from the model checkpoint at mPLUG/DocOwl2 and are newly initialized: ['model.layers.0.self_attn.rotary_emb.inv_freq', 'model.layers.1.self_attn.rotary_emb.inv_freq', 'model.layers.10.self_attn.rotary_emb.inv_freq', 'model.layers.11.self_attn.rotary_emb.inv_freq', 'model.layers.12.self_attn.rotary_emb.inv_freq', 'model.layers.13.self_attn.rotary_emb.inv_freq', 'model.layers.14.self_attn.rotary_emb.inv_freq', 'model.layers.15.self_attn.rotary_emb.inv_freq', 'model.layers.16.self_attn.rotary_emb.inv_freq', 'model.layers.17.self_attn.rotary_emb.inv_freq', 'model.layers.18.self_attn.rotary_emb.inv_freq', 'model.layers.19.self_attn.rotary_emb.inv_freq', 'model.layers.2.self_attn.rotary_emb.inv_freq', 'model.layers.20.self_attn.rotary_emb.inv_freq', 'model.layers.21.self_attn.rotary_emb.inv_freq', 'model.layers.22.self_attn.rotary_emb.inv_freq', 'model.layers.23.self_attn.rotary_emb.inv_freq', 'model.layers.24.self_attn.rotary_emb.inv_freq', 

generation_config.json:   0%|          | 0.00/162 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuratio

Tesla T4 | used 12.15 / 14.6 GiB


In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

subset: 100 страниц


In [ ]:
# DocOwl ожидает PIL.Image + текстовый промпт; в зависимости от версии
# либо есть метод model.chat(images=[img], query=prompt), либо нужно
# собрать messages вручную. Универсальный путь — через preprocessor:
from transformers import AutoProcessor
try:
    processor = AutoProcessor.from_pretrained(MODEL_REPO, trust_remote_code=True)
except Exception:
    processor = None
    print('processor не нужен — используем model.chat() напрямую')

preprocessor_config.json:   0%|          | 0.00/317 [00:00<?, ?B/s]

In [ ]:
import traceback
from pathlib import Path
from PIL import Image

out_path = Path(cfg['output']['results_dir']) / 'predictions.jsonl'
out_path.parent.mkdir(parents=True, exist_ok=True)
done = already_processed_ids(out_path)

PROMPT = cfg['inference']['prompt']
MAX_NEW = cfg['inference']['max_new_tokens']

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done: continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists(): continue
        rec = PredictionRecord(page_id=gt.page_id, model='mplug_docowl')
        try:
            img = Image.open(img_path).convert('RGB')
            with Timer('infer') as t:
                if hasattr(model, 'chat'):
                    out = model.chat(image=img, msgs=[{'role':'user','content': PROMPT}],
                                     tokenizer=tokenizer, sampling=False, max_new_tokens=MAX_NEW)
                else:
                    inputs = processor(images=img, text=PROMPT, return_tensors='pt').to(model.device)
                    with torch.no_grad():
                        gen = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False)
                    out = processor.batch_decode(gen, skip_special_tokens=True)[0]
            rec.full_text = out if isinstance(out, str) else str(out)
            rec.raw_output = rec.full_text
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
        w.write(rec.to_dict())
print('готово →', out_path)

Traceback (most recent call last):
  File "/tmp/ipykernel_11136/1420680422.py", line 22, in <cell line: 0>
    out = model.chat(image=img, msgs=[{'role':'user','content': PROMPT}],
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: MPLUGDocOwl2.chat() got an unexpected keyword argument 'image'
Traceback (most recent call last):
  File "/tmp/ipykernel_11136/1420680422.py", line 22, in <cell line: 0>
    out = model.chat(image=img, msgs=[{'role':'user','content': PROMPT}],
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: MPLUGDocOwl2.chat() got an unexpected keyword argument 'image'
Traceback (most recent call last):
  File "/tmp/ipykernel_11136/1420680422.py", line 22, in <cell line: 0>
    out = model.chat(image=img, msgs=[{'role':'user','content': PROMPT}],
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: MPLUGDocOwl2.chat() got an unexpected keyword argument 'image'
Traceback (mo

готово → results/mplug_docowl/predictions.jsonl


Traceback (most recent call last):
  File "/tmp/ipykernel_11136/1420680422.py", line 22, in <cell line: 0>
    out = model.chat(image=img, msgs=[{'role':'user','content': PROMPT}],
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: MPLUGDocOwl2.chat() got an unexpected keyword argument 'image'
Traceback (most recent call last):
  File "/tmp/ipykernel_11136/1420680422.py", line 22, in <cell line: 0>
    out = model.chat(image=img, msgs=[{'role':'user','content': PROMPT}],
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: MPLUGDocOwl2.chat() got an unexpected keyword argument 'image'
Traceback (most recent call last):
  File "/tmp/ipykernel_11136/1420680422.py", line 22, in <cell line: 0>
    out = model.chat(image=img, msgs=[{'role':'user','content': PROMPT}],
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: MPLUGDocOwl2.chat() got an unexpected keyword argument 'image'


In [ ]:
# Освобождаем GPU перед следующей моделью
del model; cuda_free(); print(gpu_info())

Tesla T4 | used 0.01 / 14.6 GiB


---
# Часть 4 · olmOCR (AllenAI)

**olmOCR** — пайплайн от AllenAI поверх Qwen2-VL-7B-Instruct, дообученный на ~250K страницах. Чекпоинт: [`allenai/olmOCR-7B-0225-preview`](https://huggingface.co/allenai/olmOCR-7B-0225-preview).

> Для OmniDocBench используется упрощённый путь без anchor-текста (в отчёте отметить, что это даёт небольшой проигрыш по сравнению с полным пайплайном).

In [ ]:
# Системные зависимости: poppler нужен для anchor-текста
!apt-get -qq install -y poppler-utils ttf-mscorefonts-installer 2>&1 | tail -1
!pip install -q olmocr

Processing triggers for man-db (2.10.2-1) ...


In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord
cfg = load_config('configs/olmocr.yaml')
cfg

{'model': {'name': 'olmocr',
  'hf_repo': 'allenai/olmOCR-7B-0225-preview',
  'trust_remote_code': False,
  'torch_dtype': 'bfloat16',
  'device_map': 'auto'},
 'inference': {'use_anchor_text': True,
  'anchor_target_length': 4000,
  'max_new_tokens': 3000,
  'temperature': 0.1,
  'pipeline_concurrency': 1},
 'dataset': {'name': 'omnidocbench', 'subset_size': 100, 'split': 'test'},
 'output': {'results_dir': 'results/olmocr', 'save_format': 'jsonl'}}

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_REPO = cfg['model']['hf_repo']
processor = AutoProcessor.from_pretrained(MODEL_REPO)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_REPO,
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.69G [00:00<?, ?B/s]

`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

Tesla T4 | used 11.87 / 14.6 GiB


In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

subset: 100 страниц


In [ ]:
import traceback, base64, io
from pathlib import Path
from PIL import Image

out_path = Path(cfg['output']['results_dir']) / 'predictions.jsonl'
out_path.parent.mkdir(parents=True, exist_ok=True)
done = already_processed_ids(out_path)

MAX_NEW = cfg['inference']['max_new_tokens']
TEMP    = cfg['inference']['temperature']

OLMOCR_PROMPT = (
    'Below is the image of one page of a document. Just return the plain text '
    'representation of this document as if you were reading it naturally. '
    'Convert equations to LaTeX and tables to HTML. Do not hallucinate.'
)

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done: continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists(): continue
        rec = PredictionRecord(page_id=gt.page_id, model='olmocr')
        try:
            img = Image.open(img_path).convert('RGB')
            messages = [{
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': OLMOCR_PROMPT},
                ],
            }]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(text=[text], images=[img], padding=True, return_tensors='pt').to(model.device)
            with Timer('infer') as t, torch.no_grad():
                gen = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=TEMP > 0,
                                     temperature=TEMP if TEMP > 0 else 1.0)
            out = processor.batch_decode(
                gen[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
            rec.full_text = out
            rec.raw_output = out
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
        w.write(rec.to_dict())
print('готово →', out_path)

Traceback (most recent call last):
  File "/tmp/ipykernel_11136/1292013599.py", line 36, in <cell line: 0>
    gen = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=TEMP > 0,
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 2252, in generate
    result = self._sample(
             ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 3251, in _sample
    outputs = self(**model_inputs, return_dict=True)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^

готово → results/olmocr/predictions.jsonl


Traceback (most recent call last):
  File "/tmp/ipykernel_11136/1292013599.py", line 36, in <cell line: 0>
    gen = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=TEMP > 0,
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 2252, in generate
    result = self._sample(
             ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 3251, in _sample
    outputs = self(**model_inputs, return_dict=True)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
# Освобождаем GPU перед следующей моделью
del model; cuda_free(); print(gpu_info())

Tesla T4 | used 0.09 / 14.6 GiB


---
# Часть 5 · MonkeyOCR

**MonkeyOCR** — модель с парадигмой **Structure → Recognition → Relation**. Чекпоинт: [`echo840/MonkeyOCR`](https://huggingface.co/echo840/MonkeyOCR).

In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord
cfg = load_config('configs/monkeyocr.yaml')
cfg

{'model': {'name': 'monkeyocr',
  'hf_repo': 'echo840/MonkeyOCR',
  'trust_remote_code': True,
  'torch_dtype': 'bfloat16',
  'device_map': 'auto'},
 'inference': {'task': 'full_page',
  'prompt': None,
  'max_new_tokens': 6144,
  'do_sample': False},
 'dataset': {'name': 'omnidocbench', 'subset_size': 100, 'split': 'test'},
 'output': {'results_dir': 'results/monkeyocr', 'save_format': 'jsonl'}}

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
MODEL_REPO = cfg['model']['hf_repo']
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

ValueError: Unrecognized model in echo840/MonkeyOCR. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: albert, align, altclip, audio-spectrogram-transformer, autoformer, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, blenderbot, blenderbot-small, blip, blip-2, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, conditional_detr, convbert, convnext, convnextv2, cpmant, ctrl, cvt, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deformable_detr, deit, depth_anything, deta, detr, dinat, dinov2, distilbert, donut-swin, dpr, dpt, efficientformer, efficientnet, electra, encodec, encoder-decoder, ernie, ernie_m, esm, falcon, falcon_mamba, fastspeech2_conformer, flaubert, flava, fnet, focalnet, fsmt, funnel, fuyu, gemma,

In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

In [ ]:
# MonkeyOCR имеет встроенный метод model.chat_full_page(...) (см. README модели).
# Если интерфейс изменится — заменить на актуальный.
import traceback
from pathlib import Path
from PIL import Image

out_path = Path(cfg['output']['results_dir']) / 'predictions.jsonl'
out_path.parent.mkdir(parents=True, exist_ok=True)
done = already_processed_ids(out_path)

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done: continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists(): continue
        rec = PredictionRecord(page_id=gt.page_id, model='monkeyocr')
        try:
            img = Image.open(img_path).convert('RGB')
            with Timer('infer') as t, torch.no_grad():
                if hasattr(model, 'chat_full_page'):
                    out = model.chat_full_page(tokenizer, img,
                                               max_new_tokens=cfg['inference']['max_new_tokens'])
                else:
                    out = model.chat(tokenizer, img,
                                     query='Convert this page to markdown',
                                     max_new_tokens=cfg['inference']['max_new_tokens'])
            rec.full_text = out if isinstance(out, str) else str(out)
            rec.raw_output = rec.full_text
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
        w.write(rec.to_dict())
print('готово →', out_path)

In [ ]:
# Освобождаем GPU
del model; cuda_free(); print(gpu_info())